# Análise Exploratória de Dados (EDA)

Grupo 2

João Costa | 113564

Sílvia Ribeiro | 129428

Vitória Rodrigues | 130557

A análise exploratória de dados (EDA) tem como objetivo compreender as dinâmicas de afluência nos POIs monitorizados na ilha de São Miguel. Esta etapa permite caracterizar a estrutura do dataset, identificar padrões relevantes e avaliar a influência de variáveis contextuais sobre a variável dependente *total_incoming*.

A análise estrutura-se em quatro momentos principais: 
1. Caracterização do Dataset
2. Padrões Temporais
3. Condições Meteorológicas
4. Análise de Correlação



In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

##  Carregamento dos datasets


In [ ]:
START_DIR = Path.cwd().resolve()
print(f"Diretoria atual de execução: {START_DIR}")

DATA_DIR = None
for root in [START_DIR, *START_DIR.parents]:
    candidate = root / "dataset"
    if candidate.exists() and candidate.is_dir():
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta central 'dataset'. "
        "Certifica-te de que manténs a pasta 'dataset' na raiz do projeto."
    )

print(f"Pasta unificada de dados encontrada em: {DATA_DIR}")

EDA_PATH = DATA_DIR / "df_preprocessed_eda.csv"

if not EDA_PATH.exists():
    raise FileNotFoundError(
        f"O ficheiro para análise não foi encontrado: {EDA_PATH}. "
        "Certifica-te de que executaste o Notebook 2 com sucesso."
    )

df = pd.read_csv(EDA_PATH)
df["movement_date"] = pd.to_datetime(df["movement_date"], errors="coerce")
df = df.sort_values(["movement_date", "geography_key"]).reset_index(drop=True)

print(f"Dataset de Análise Exploratória carregado com sucesso! Shape: {df.shape}")
display(df.head())

## 1. Caracterização estatística do dataset

O objetivo desta análise é compreender a estrutura geral do dataset, avaliando a sua dimensão, qualidade e distribuição das principais variáveis, bem como dos missing values.

In [ ]:
def inspect_dataframe(df, name, head_rows=5):
    print(f"{name} - shape: {df.shape}")
    display(df.head(head_rows))
    df.info()
    print("\nValores em falta (%):")
    display((df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct"))

inspect_dataframe(df, "Dataset pre-processado para EDA")

print("Numero de POIs presentes:", df["geography_key"].nunique())
print("POIs presentes:")
display(df[["geography_key", "poi_name"]].drop_duplicates().sort_values("poi_name"))
print("Cobertura temporal:", df["movement_date"].min().date(), "a", df["movement_date"].max().date())


## Perfis comparativos entre POIs

De forma a apoiar a análise exploratória, procede-se à síntese de métricas por POI, permitindo comparar a afluência média, a variabilidade e a frequência de dias com precipitação, contribuindo para a caracterização dos diferentes perfis de visitação.

In [ ]:
poi_profile = (
    df.groupby("poi_name", dropna=False)
      .agg(
          media_incoming=("total_incoming", "mean"),
          desvio_incoming=("total_incoming", "std"),
          media_temperatura=("weather_temperature_2m_mean", "mean"),
          precipitacao_media=("weather_precipitation_sum", "mean"),
          proporcao_dias_com_precipitacao=("has_precipitation", "mean"),
      )
      .sort_values("media_incoming", ascending=False)
      .round(3)
)
display(poi_profile)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

poi_profile["media_incoming"].sort_values().plot(kind="barh", ax=axes[0], color="#3A86FF")
axes[0].set_title("Media de total_incoming por POI")
axes[0].set_xlabel("Media de total_incoming")
axes[0].set_ylabel("POI")

poi_profile["proporcao_dias_com_precipitacao"].sort_values().plot(kind="barh", ax=axes[1], color="#219EBC")
axes[1].set_title("Proporcao de dias com precipitacao por POI")
axes[1].set_xlabel("Proporcao")
axes[1].set_ylabel("POI")

plt.tight_layout()
plt.show()

## Resumo geral por POI

Dado que o estudo abrange múltiplos pontos de interesse (POIs), torna-se relevante realizar uma análise comparativa entre os mesmos, considerando a cobertura temporal, os níveis médios de afluência, a variabilidade dos dados e as condições meteorológicas associadas.

In [ ]:
summary_by_poi = (
    df.groupby(["geography_key", "poi_name"], dropna=False)
      .agg(
          n_observacoes=("total_incoming", "size"),
          primeira_data=("movement_date", "min"),
          ultima_data=("movement_date", "max"),
          media_incoming=("total_incoming", "mean"),
          mediana_incoming=("total_incoming", "median"),
          desvio_incoming=("total_incoming", "std"),
          min_incoming=("total_incoming", "min"),
          max_incoming=("total_incoming", "max"),
          media_temperatura=("weather_temperature_2m_mean", "mean"),
          precipitacao_media=("weather_precipitation_sum", "mean"),
      )
      .sort_values("media_incoming", ascending=False)
      .round(2)
      .reset_index()
)

display(summary_by_poi)

## Distribuição da variável alvo

A variável alvo do projeto é *total_incoming*. Nesta secção analisa-se a sua distribuição a nível global e por POI, com o objetivo de identificar assimetrias, avaliar a variabilidade dos dados e detetar possíveis diferenças estruturais entre os diferentes locais.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["total_incoming"], bins=12, kde=True, ax=axes[0], color="#2E8B57")
axes[0].set_title("Distribuicao global de total_incoming")
axes[0].set_xlabel("total_incoming")

sns.boxplot(data=df, x="total_incoming", ax=axes[1], color="#87CEEB")
axes[1].set_title("Boxplot global de total_incoming")
axes[1].set_xlabel("total_incoming")

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="poi_name", y="total_incoming", color="#A8DADC")
plt.title("Distribuicao de total_incoming por POI")
plt.xlabel("POI")
plt.ylabel("total_incoming")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 2. Evolução temporal da afluência

Atendendo ao caráter preditivo do projeto, a dimensão temporal constitui um elemento central da análise. Nesta secção, analisa-se a evolução diária da afluência, considerando tanto o comportamento agregado como as variações específicas por POI.

In [ ]:
daily_total = (
    df.groupby("movement_date", as_index=False)["total_incoming"]
      .sum()
      .sort_values("movement_date")
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=daily_total, x="movement_date", y="total_incoming", marker="o", color="#C44E52")
plt.title("Evolucao diaria da afluencia total")
plt.xlabel("Data")
plt.ylabel("Soma diaria de total_incoming")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 6))
sns.lineplot(data=df, x="movement_date", y="total_incoming", hue="poi_name", marker="o")
plt.title("Evolucao diaria de total_incoming por POI")
plt.xlabel("Data")
plt.ylabel("total_incoming")
plt.xticks(rotation=30)
plt.legend(title="POI", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Padrões de calendário

A estrutura temporal construída na fase de pré-processamento permite analisar a variação da afluência em função de diferentes componentes do calendário, nomeadamente o dia da semana, a distinção entre dias úteis e fins de semana, o mês e a presença de feriados, permitindo identificar padrões comportamentais associados ao tempo.

In [ ]:
day_order = ["Segunda", "Terca", "Quarta", "Quinta", "Sexta", "Sabado", "Domingo"]
weekday_summary = (
    df.groupby("day_name", dropna=False)["total_incoming"]
      .agg(["count", "mean", "median"])
      .reindex(day_order)
      .round(2)
)
print("Resumo por dia da semana:")
display(weekday_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df, x="day_name", y="total_incoming", order=day_order, estimator=np.mean, errorbar=None, ax=axes[0], color="#6A5ACD")
axes[0].set_title("Media de total_incoming por dia da semana")
axes[0].set_xlabel("Dia da semana")
axes[0].set_ylabel("Media de total_incoming")
axes[0].tick_params(axis="x", rotation=25)

weekend_labels = {False: "Dia util", True: "Fim de semana"}
weekend_plot = df.assign(tipo_dia=df["is_weekend"].map(weekend_labels))
sns.boxplot(data=weekend_plot, x="tipo_dia", y="total_incoming", ax=axes[1], color="#F4A261")
axes[1].set_title("Distribuicao da afluencia: dias uteis vs fim de semana")
axes[1].set_xlabel("Tipo de dia")
axes[1].set_ylabel("total_incoming")

plt.tight_layout()
plt.show()

holiday_summary = (
    df.groupby("is_holiday")["total_incoming"]
      .agg(["count", "mean", "median"])
      .round(2)
)
print("Resumo por condicao de feriado:")
display(holiday_summary)


## 3. Relação entre afluência e condições meteorológicas

Os dados meteorológicos foram integrados como variáveis contextuais diárias à escala da ilha de São Miguel. Nesta secção procede-se à análise da relação entre a afluência e os principais indicadores climatéricos — temperatura, precipitação, velocidade do vento e humidade relativa —, com o objetivo de avaliar o seu contributo para a explicação do comportamento dos visitantes.

In [ ]:
weather_cols = [
    "weather_temperature_2m_mean",
    "weather_precipitation_sum",
    "weather_wind_speed_10m_mean",
    "weather_relative_humidity_2m_mean",
    "weather_temperature_range",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

plot_specs = [
    ("weather_temperature_2m_mean", "Temperatura media (2m)", "#E76F51"),
    ("weather_precipitation_sum", "Precipitacao diaria", "#457B9D"),
    ("weather_wind_speed_10m_mean", "Velocidade media do vento (10m)", "#2A9D8F"),
    ("weather_relative_humidity_2m_mean", "Humidade relativa media", "#8D99AE"),
]

for ax, (col, label, color) in zip(axes, plot_specs):
    sns.scatterplot(data=df, x=col, y="total_incoming", hue="poi_name", ax=ax, s=70)
    sns.regplot(data=df, x=col, y="total_incoming", scatter=False, ax=ax, color=color, line_kws={"linewidth": 1.5})
    ax.set_title(f"total_incoming vs {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("total_incoming")
    if ax is not axes[0]:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    axes[0].legend(handles=handles, labels=labels, title="POI", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

precip_summary = (
    df.groupby("has_precipitation")["total_incoming"]
      .agg(["count", "mean", "median"])
      .round(2)
)
print("Comparacao entre dias com e sem precipitacao:")
display(precip_summary)

## 4. Correlações entre variáveis numéricas

A matriz de correlação permite avaliar a intensidade e direção das relações lineares entre a variável alvo e as restantes variáveis numéricas relevantes. Esta análise contribui para a compreensão da estrutura dos dados, bem como para a identificação de variáveis com maior potencial explicativo, apoiando de forma fundamentada a seleção de atributos para a fase de modelação.

In [ ]:
correlation_cols = [
    "total_incoming",
    "incoming_record_count",
    "incoming_source_rows",
    "day_of_week",
    "day_of_month",
    "day_of_year",
    "week_of_year",
    "month_number",
    "weather_hourly_records",
    "weather_temperature_2m_mean",
    "weather_precipitation_sum",
    "weather_wind_speed_10m_mean",
    "weather_relative_humidity_2m_mean",
    "weather_temperature_range",
    "days_since_first_observation",
    "log_total_incoming",
    "log_weather_precipitation_sum",
]

corr_df = df[correlation_cols].corr(numeric_only=True)

target_corr = (
    corr_df[["total_incoming"]]
      .drop(index="total_incoming")
      .sort_values("total_incoming", ascending=False)
      .round(3)
)
print("Correlacao das variaveis numericas com total_incoming:")
display(target_corr)

plt.figure(figsize=(12, 8))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=False)
plt.title("Matriz de correlacao das principais variaveis numericas")
plt.tight_layout()
plt.show()

## Conclusão 

A análise exploratória permitiu caracterizar a variabilidade da afluência, identificar padrões temporais e evidenciar a influência das condições meteorológicas no comportamento dos visitantes.

Os resultados confirmam o potencial explicativo das variáveis temporais e climatéricas, bem como a existência de heterogeneidade entre POIs.

Deste modo, a EDA fornece uma base sólida para a fase de modelação preditiva, garantindo que as decisões metodológicas são sustentadas por evidência empírica.